# Survival analysis: time-to-event models: worked solution

Worked solution for the exercise in the lesson **[Survival analysis: time-to-event models](https://ramkumarmd.com/medical-ai/machine-learning/survival-analysis)** (*Machine learning* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](05-survival-analysis.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** Survival analysis models the time until an event, such as kidney failure, while correctly handling patients whose follow-up ended before any event (censoring). Simply counting events among everyone underestimates risk because some patients have not been followed long enough. Kaplan-Meier curves estimate risk over time, Cox models relate risk to patient characteristics, and both can give a predicted risk at a chosen time, such as five years.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q lifelines scikit-survival

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### A synthetic kidney-failure cohort

Synthetic time to kidney failure with censoring at varying follow-up times.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(21)
n = 3000
df = pd.DataFrame({
    "age": rng.integers(40, 80, n),
    "egfr": rng.normal(60, 18, n).clip(15, 120).round(),   # mL/min/1.73m²
    "uacr_high": rng.binomial(1, 0.35, n),                  # albuminuria
    "sglt2i": rng.binomial(1, 0.4, n),
})
hazard = 0.02 * np.exp(0.02 * (df.age - 60) - 0.04 * (df.egfr - 60)
                       + 0.8 * df.uacr_high - 0.3 * df.sglt2i)
event_time = rng.exponential(1 / hazard)
censor_time = rng.uniform(1, 8, n)           # end of follow-up varies by patient
df["years"] = np.minimum(event_time, censor_time).round(2)
df["kidney_failure"] = (event_time <= censor_time).astype(int)
print(f"{df.kidney_failure.sum()} events among {n} patients; "
      f"{(1 - df.kidney_failure.mean()):.0%} censored")

# 368 events among 3000 patients; 88% censored

### Kaplan-Meier: risk over time

Continuing from the previous block. Install lifelines first if needed: pip install lifelines

In [ ]:
from lifelines import KaplanMeierFitter

naive = df[df.years <= 5].kidney_failure.sum() / n
print(f"Naive 'events / everyone' by 5 years: {naive:.1%}")

kmf = KaplanMeierFitter().fit(df.years, df.kidney_failure)
print(f"Kaplan-Meier risk by 5 years:        {1 - kmf.predict(5):.1%}")

# Naive 'events / everyone' by 5 years: 10.9%
# Kaplan-Meier risk by 5 years:        13.8%

### Cox proportional-hazards regression

Continuing from the previous block: fit a Cox model and show hazard ratios with 95% confidence intervals.

In [ ]:
from lifelines import CoxPHFitter

cph = CoxPHFitter().fit(df, duration_col="years", event_col="kidney_failure")
print(cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%"]].round(2))
print(f"\nConcordance index {cph.concordance_index_:.3f}")

#            exp(coef)  exp(coef) lower 95%  exp(coef) upper 95%
# covariate
# age             1.02                 1.01                 1.02
# egfr            0.96                 0.95                 0.96
# uacr_high       2.77                 2.25                 3.41
# sglt2i          0.72                 0.58                 0.89
#
# Concordance index 0.737

### From hazard ratios to individual risk

Continuing from the previous block: predicted 5-year risk for two synthetic patients.

In [ ]:
patients = pd.DataFrame({"age": [55, 70], "egfr": [80, 35],
                         "uacr_high": [0, 1], "sglt2i": [0, 0]})
risk_5y = 1 - cph.predict_survival_function(patients, times=[5]).iloc[0]
print(risk_5y.round(3).to_string())

# 0    0.034
# 1    0.559

## Exercise

**Goal:** Analyse a synthetic kidney-failure cohort with survival methods, and see how censoring and competing risks change the answer.

**Data:** Synthetic cohort generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the naive and Kaplan-Meier five-year risks, and plot Kaplan-Meier curves by albuminuria status.

In [ ]:
import matplotlib.pyplot as plt

print(f"Naive 5-year risk {naive:.1%}; Kaplan-Meier 5-year risk {1 - kmf.predict(5):.1%}")

fig, ax = plt.subplots(figsize=(6, 4))
for value, label in [(0, "No albuminuria"), (1, "Albuminuria")]:
    group = df[df.uacr_high == value]
    KaplanMeierFitter().fit(group.years, group.kidney_failure, label=label).plot_survival_function(ax=ax)
ax.set(xlabel="Years", ylabel="Free of kidney failure", title="Kaplan-Meier curves (synthetic cohort)")
plt.show()

The naive estimate is too low because patients censored early (followed for less than five years) are
counted as if they could never have an event. Kaplan-Meier uses each patient only for the time they were
actually observed.

### Step 2

Fit the Cox model, check the proportional-hazards assumption, and report hazard ratios with confidence intervals.

In [ ]:
print(cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].round(3))
_ = cph.check_assumptions(df, p_value_threshold=0.05, show_plots=False)   # prints its advice

Compare with the values used in the simulation: age exp(0.02) = 1.02 per year, eGFR exp(−0.04) = 0.96 per
mL/min/1.73m², albuminuria exp(0.8) = 2.23 and SGLT2 inhibitor exp(−0.3) = 0.74. Three estimates are close;
albuminuria's 2.77 is higher, with the true 2.23 just at the lower end of its confidence interval. That is
what sampling variation looks like with a few hundred events: a different seed gives a different estimate.
The data were simulated with constant hazard ratios, and the proportional-hazards check finds no problem;
with real data, a flagged variable is often handled by stratifying on it or letting its effect vary with time.

### Step 3

Add a competing risk of death to the simulation and compare 1 − Kaplan-Meier with the Aalen-Johansen cumulative incidence of kidney failure.

In [ ]:
from lifelines import AalenJohansenFitter

rng = np.random.default_rng(22)
death_hazard = 0.03 * np.exp(0.06 * (df.age - 60))         # older patients die sooner, from other causes
death_time = rng.exponential(1 / death_hazard)
first = np.minimum.reduce([event_time, death_time, censor_time])
cr = pd.DataFrame({"years": first,               # unrounded: Aalen-Johansen needs untied times
                   "event": np.select([first == event_time, first == death_time], [1, 2], 0)})
print(cr.event.value_counts().rename({0: "censored", 1: "kidney failure", 2: "death"}).to_string())

km_cr = KaplanMeierFitter().fit(cr.years, cr.event == 1)          # treats deaths as censored: wrong here
aj = AalenJohansenFitter(seed=0).fit(cr.years, cr.event, event_of_interest=1)
cif_5 = aj.cumulative_density_.loc[:5].iloc[-1, 0]
print(f"\n1 − Kaplan-Meier at 5 years:     {1 - km_cr.predict(5):.1%}")
print(f"Aalen-Johansen at 5 years:       {cif_5:.1%}")

1 − Kaplan-Meier answers "what would the risk be if nobody could die of anything else", which is not the
risk a patient faces. Patients who die cannot go on to kidney failure, so the Aalen-Johansen cumulative
incidence is lower, and the gap grows with the death rate (here, with age).

### Step 4

Calculate the C-index on a held-out test set and draw a calibration plot of predicted versus observed five-year risk.

In [ ]:
from lifelines.utils import concordance_index
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.3, random_state=0)
cox = CoxPHFitter().fit(train, duration_col="years", event_col="kidney_failure")
risk_score = cox.predict_partial_hazard(test)
print(f"Held-out C-index {concordance_index(test.years, -risk_score, test.kidney_failure):.3f}")

pred_5y = 1 - cox.predict_survival_function(test, times=[5]).iloc[0].to_numpy()
test = test.assign(pred_5y=pred_5y, group=pd.qcut(pred_5y, 5, labels=False))
rows = []
for g, d in test.groupby("group"):
    km = KaplanMeierFitter().fit(d.years, d.kidney_failure)
    rows.append({"predicted": d.pred_5y.mean(), "observed": 1 - km.predict(5)})
cal = pd.DataFrame(rows)
print((cal * 100).round(1).rename_axis("risk fifth").to_string())

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot(cal.predicted, cal.observed, "o-")
ax.plot([0, cal.max().max()], [0, cal.max().max()], ":", color="grey")
ax.set(xlabel="Predicted 5-year risk", ylabel="Observed (Kaplan-Meier) 5-year risk", title="Calibration at 5 years")
plt.show()

Observed risk in each group is estimated with Kaplan-Meier, not by counting events, because many test
patients were censored before five years. Predicted and observed risks rise together, but with about 900
test patients each fifth has only a couple of dozen events, so individual groups can miss by a few points.

### Step 5

Fit a random survival forest (scikit-survival) and compare its C-index with the Cox model.

In [ ]:
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

X_cols = ["age", "egfr", "uacr_high", "sglt2i"]
y_train = Surv.from_arrays(train.kidney_failure.astype(bool), train.years)
rsf = RandomSurvivalForest(n_estimators=300, min_samples_leaf=15, n_jobs=-1, random_state=0)
rsf.fit(train[X_cols], y_train)
rsf_risk = rsf.predict(test[X_cols])
print(f"Random survival forest C-index {concordance_index(test.years, -rsf_risk, test.kidney_failure):.3f}")
print(f"Cox model C-index              {concordance_index(test.years, -risk_score, test.kidney_failure):.3f}")

The data follow a Cox model exactly, so the forest has nothing extra to learn and does no better (often a
little worse). Flexible survival models earn their place when effects are non-linear or interacting, and
they should be compared on calibration as well as the C-index.

---

Lesson: [https://ramkumarmd.com/medical-ai/machine-learning/survival-analysis](https://ramkumarmd.com/medical-ai/machine-learning/survival-analysis). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.